In [ ]:
# install  (Runtime > Change runtime type > T4 GPU; restart runtime after this cell)
!pip uninstall -y torchao
!pip install -q -U transformers peft accelerate gradio pypdf huggingface_hub


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 37.5 MB/s eta 0:00:00


In [ ]:
# HF login (Colab secret: HF_TOKEN, with notebook access enabled)
from google.colab import userdata
from huggingface_hub import login, HfApi

HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)
HF_USER = HfApi().whoami()["name"]
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
EXTRACTOR = f"{HF_USER}/pdf-extractor-qwen-1.5b"
STRUCTURER = f"{HF_USER}/pdf-structurer-qwen-1.5b"
print(EXTRACTOR, "|", STRUCTURER)


developertce/pdf-extractor-qwen-1.5b | developertce/pdf-structurer-qwen-1.5b


In [ ]:
# load base model once + both adapters
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
tok = AutoTokenizer.from_pretrained(BASE, padding_side="left")
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE).to(DEVICE)
model = PeftModel.from_pretrained(base, EXTRACTOR, adapter_name="extractor")
model.load_adapter(STRUCTURER, adapter_name="structurer")
model.eval()
print("loaded on", DEVICE)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

adapter_model.safetensors: downloading bytes:           |  0.00B            

adapter_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 73.9MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

loaded on cuda


In [ ]:
# pipeline: PDF -> chunks -> extractor -> structurer -> JSON
import json, re
from pypdf import PdfReader

EXTRACT_SYSTEM = (
    "Extract from the text only the passages relevant to the given topic. "
    "Copy them verbatim, separated by blank lines. If nothing is relevant, reply exactly: NONE"
)
STRUCT_SYSTEM = (
    "Convert the extracted passages about a topic into structured JSON. Use only information in the passages. "
    "Output only a JSON object with keys: "
    "title (string), summary (2-3 sentences), key_points (list of 3-6 short strings), "
    "entities (list of {\"name\": string, \"type\": string}), dates (list of strings, empty if none). "
    "No text outside the JSON."
)
KEYS = {"title", "summary", "key_points", "entities", "dates"}
EXTRACT_BS = 8 if DEVICE == "cuda" else 1
EXTRACT_MAX_TOKENS = 1000   # long verbatim copies need room
STRUCT_GROUP_CHARS = 1800   # smaller groups suit a structurer trained on short passages


def prompt(system, user):
    return tok.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        add_generation_prompt=True, tokenize=False,
    )


@torch.no_grad()
def generate(adapter, prompts, max_new_tokens):
    model.set_adapter(adapter)
    enc = tok(prompts, return_tensors="pt", padding=True).to(DEVICE)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    n = enc["input_ids"].shape[1]
    return [tok.decode(o[n:], skip_special_tokens=True).strip() for o in out]


def is_none(s):
    return s.strip().rstrip(".").strip().upper() == "NONE"


def read_pdf(path):
    return "\n\n".join((p.extract_text() or "") for p in PdfReader(path).pages)


def chunk_text(text, max_chars=3000):
    paras = []
    for p in re.split(r"\n\s*\n", text):
        p = re.sub(r"\s*\n\s*", " ", p).strip()
        if not p:
            continue
        while len(p) > 1500:  # split very long paragraphs at a sentence boundary
            cut = p.rfind(". ", 0, 1500)
            cut = cut + 1 if cut > 300 else 1500
            paras.append(p[:cut].strip())
            p = p[cut:].strip()
        paras.append(p)
    chunks, cur = [], ""
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = f"{cur}\n\n{p}" if cur else p
    if cur:
        chunks.append(cur)
    return chunks


def parse_json(txt):
    try:
        obj = json.loads(txt[txt.index("{"): txt.rindex("}") + 1])
        return obj if KEYS <= set(obj) else {"error": "missing keys", "raw": txt}
    except Exception:
        return {"error": "invalid json", "raw": txt}


def merge(results):
    ok = [r for r in results if "error" not in r]
    if not ok:
        return {"error": "structurer returned invalid JSON", "raw": results}
    uniq = lambda items, key: list({key(i): i for i in items}.values())
    return {
        "title": ok[0]["title"],
        "summary": " ".join(r["summary"] for r in ok),
        "key_points": uniq([k for r in ok for k in r["key_points"]], lambda k: str(k).lower()),
        "entities": uniq(
            [e for r in ok for e in r["entities"]],
            lambda e: str(e.get("name", "")).lower() if isinstance(e, dict) else str(e).lower(),
        ),
        "dates": uniq([d for r in ok for d in r["dates"]], lambda d: str(d).lower()),
    }


def run_pipeline(pdf_path, topic, progress=None):
    tick = progress or (lambda *a, **k: None)
    text = read_pdf(pdf_path)
    if not text.strip():
        return "No extractable text (scanned PDF?).", None, ""
    chunks = chunk_text(text)
    found = []
    for i in range(0, len(chunks), EXTRACT_BS):
        tick(i / len(chunks), desc=f"Extracting {i}/{len(chunks)} chunks")
        outs = generate(
            "extractor",
            [prompt(EXTRACT_SYSTEM, f"Topic: {topic}\n\nText:\n{c}") for c in chunks[i:i + EXTRACT_BS]],
            EXTRACT_MAX_TOKENS,
        )
        found += [o for o in outs if o and not is_none(o)]
    found = list(dict.fromkeys(found))
    status = f"{len(chunks)} chunks, {len(found)} with relevant content"
    if not found:
        return status, {"message": "No content related to this topic found in the PDF."}, ""

    groups, cur = [], ""
    for f in found:
        if cur and len(cur) + len(f) > STRUCT_GROUP_CHARS:
            groups.append(cur)
            cur = f
        else:
            cur = f"{cur}\n\n{f}" if cur else f
    groups.append(cur)

    results = []
    for j, g in enumerate(groups):
        tick(0.9, desc=f"Structuring {j + 1}/{len(groups)}")
        out = generate("structurer", [prompt(STRUCT_SYSTEM, f"Topic: {topic}\n\nPassages:\n{g}")], 450)[0]
        results.append(parse_json(out))
    return status, merge(results), "\n\n---\n\n".join(found)


In [ ]:
# web UI (public link printed below)
import gradio as gr


def ui(pdf, topic, progress=gr.Progress()):
    if pdf is None or not topic.strip():
        return "Upload a PDF and enter a topic.", None, ""
    return run_pipeline(pdf, topic.strip(), progress)


demo = gr.Interface(
    fn=ui,
    inputs=[gr.File(label="PDF", file_types=[".pdf"], type="filepath"), gr.Textbox(label="Topic")],
    outputs=[gr.Textbox(label="Status"), gr.JSON(label="Structured output"), gr.Textbox(label="Extracted passages", lines=12)],
    title="PDF Topic Extractor + Structurer",
    flagging_mode="never",
)
demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b0cbd6b6c79b137ba3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
